# Ejercicio 7 - Indicadores

Este notebook ejecuta las consultas de `sql/07_indicadores/` sobre DuckDB y muestra la tabla que alimenta cada
indicador. La **visualizacion** se hace en Metabase (la herramienta del ambiente): el tablero se construye con
`python scripts/metabase_dashboard.py`, que usa exactamente estas consultas sobre la base materializada
`data/processed/taxis.duckdb`. Captura del tablero: `docs/dashboard/tablero.png`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import matplotlib.pyplot as plt
import pandas as pd
from lab_db import DIR_SQL, connect, leer_consulta, run_query_file

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
con = connect()   # conexion en memoria + vistas de sql/00_views.sql sobre los Parquet

### `00a_kpi_viajes.sql`

- **Pregunta:** Cuantos viajes validos hay en el periodo analizado?
- **Indicador:** KPI - total de viajes validos (millones).
- **Fuente:** trips_clean

In [2]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "00a_kpi_viajes.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

1.87 s, 1 filas


,viajes_millones
0,68.6


### `00b_kpi_facturacion.sql`

- **Pregunta:** Cuanto se facturo en total en el periodo analizado?
- **Indicador:** KPI - facturacion total (millones de USD).
- **Fuente:** trips_clean

In [3]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "00b_kpi_facturacion.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

0.62 s, 1 filas


,facturacion_millones_usd
0,2003.3


### `01_viajes_mensuales.sql`

- **Pregunta:** Como evoluciona la demanda de taxis mes a mes y por tipo?
- **Indicador:** Viajes validos por mes y tipo de taxi.
- **Visualizacion:** lineas (x = mes, series = tipo).
- **Fuente:** trips_clean

In [4]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "01_viajes_mensuales.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

0.63 s, 40 filas


,mes,taxi_type,viajes
0,2024-01-01,green,52735
1,2024-01-01,yellow,2859076
2,2024-02-01,green,49908
3,2024-02-01,yellow,2891284
4,2024-03-01,green,53581
5,2024-03-01,yellow,3427645
6,2024-04-01,green,52347
7,2024-04-01,yellow,3402039
8,2024-05-01,green,56818
9,2024-05-01,yellow,3603557


### `02_ticket_mediano.sql`

- **Pregunta:** Cuanto paga un pasajero en un viaje tipico y como cambia en el tiempo?
- **Indicador:** Total cobrado mediano por viaje (USD), por mes y tipo.
- **Visualizacion:** lineas.
- **Fuente:** trips_clean

In [5]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "02_ticket_mediano.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

1.10 s, 40 filas


,mes,taxi_type,ticket_mediano_usd
0,2024-01-01,green,18.36
1,2024-01-01,yellow,20.16
2,2024-02-01,green,18.45
3,2024-02-01,yellow,20.46
4,2024-03-01,green,18.50
5,2024-03-01,yellow,20.65
6,2024-04-01,green,18.90
7,2024-04-01,yellow,21.00
8,2024-05-01,green,19.85
9,2024-05-01,yellow,21.60


### `03_uso_efectivo.sql`

- **Pregunta:** Se esta dejando de usar el efectivo?
- **Indicador:** % de viajes pagados en efectivo, sobre los viajes con metodo de pago informado (codigos 1-6).
- **Visualizacion:** lineas.
- **Fuente:** trips_clean

In [6]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "03_uso_efectivo.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

0.65 s, 40 filas


,mes,taxi_type,pct_efectivo
0,2024-01-01,green,30.43
1,2024-01-01,yellow,15.28
2,2024-02-01,green,29.63
3,2024-02-01,yellow,14.34
4,2024-03-01,green,30.73
5,2024-03-01,yellow,14.82
6,2024-04-01,green,28.04
7,2024-04-01,yellow,14.82
8,2024-05-01,green,28.03
9,2024-05-01,yellow,14.82


### `04_propina_tarjeta.sql`

- **Pregunta:** Que tan generosos son los pasajeros que pagan con tarjeta?
- **Indicador:** Propina mediana como % de la tarifa base en pagos con tarjeta, por anio y tipo.
- **Visualizacion:** barras agrupadas.
- **Fuente:** trips_clean (payment_type = 1)

In [7]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "04_propina_tarjeta.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

1.25 s, 4 filas


,anio,taxi_type,propina_mediana_pct
0,2024,green,23.49
1,2024,yellow,25.88
2,2026,green,23.64
3,2026,yellow,26.41


### `05_velocidad_por_hora.sql`

- **Pregunta:** A que hora es mas lenta la ciudad y cambio la congestion entre anios?
- **Indicador:** Velocidad mediana (mph) de taxis amarillos por hora del dia, lunes a viernes, por anio.
- **Visualizacion:** lineas (x = hora, series = anio).
- **Fuente:** trips_clean (taxi_type = 'yellow', isodow <= 5)

In [8]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "05_velocidad_por_hora.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

0.76 s, 48 filas


,hora,anio,velocidad_mediana_mph
0,0,2024,13.42
1,0,2026,13.40
2,1,2024,14.11
3,1,2026,14.35
4,2,2024,14.29
5,2,2026,14.86
6,3,2024,15.08
7,3,2026,15.52
8,4,2024,17.30
9,4,2026,16.79


### `06_cargos_congestion.sql`

- **Pregunta:** Cuanto pesan los cargos por congestion en lo que paga el pasajero?
- **Indicador:** % de la facturacion de taxis amarillos que corresponde al recargo de congestion y al cargo CBD, por mes.
- **Visualizacion:** area apilada.
- **Fuente:** trips_clean (taxi_type = 'yellow')

In [9]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "06_cargos_congestion.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

0.59 s, 20 filas


,mes,pct_recargo_congestion,pct_cargo_cbd
0,2024-01-01,8.18,0.00
1,2024-02-01,8.15,0.00
2,2024-03-01,7.48,0.00
3,2024-04-01,7.31,0.00
4,2024-05-01,7.16,0.00
5,2024-06-01,7.19,0.00
6,2024-07-01,7.21,0.00
7,2024-08-01,7.15,0.00
8,2024-09-01,6.88,0.00
9,2024-10-01,7.18,0.00


### `07_aeropuertos.sql`

- **Pregunta:** Que tan dependientes son los taxis amarillos de los viajes de aeropuerto?
- **Indicador:** % de viajes y % de facturacion de taxis amarillos con origen o destino en JFK, LaGuardia o Newark, por anio.
- **Visualizacion:** barras agrupadas.
- **Fuente:** trips_clean + zones

In [10]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "07_aeropuertos.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

0.84 s, 2 filas


,anio,pct_viajes,pct_facturacion
0,2024,10.07,27.80
1,2026,8.11,20.94


### `08_top_zonas_origen.sql`

- **Pregunta:** Donde se concentra la demanda de taxis?
- **Indicador:** 10 zonas con mas recogidas (ambos tipos, todo el periodo) y su % del total.
- **Visualizacion:** barras horizontales.
- **Fuente:** trips_clean + zones

In [11]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "08_top_zonas_origen.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

1.43 s, 10 filas


,zona,viajes,pct_del_total
0,Upper East Side South (Manhattan),3118696,4.54
1,Midtown Center (Manhattan),3027360,4.41
2,JFK Airport (Queens),2961000,4.32
3,Upper East Side North (Manhattan),2804357,4.09
4,Midtown East (Manhattan),2241856,3.27
5,Penn Station/Madison Sq West (Manhattan),2189484,3.19
6,Times Sq/Theatre District (Manhattan),2149868,3.13
7,Lincoln Square East (Manhattan),2089278,3.04
8,LaGuardia Airport (Queens),1969166,2.87
9,Murray Hill (Manhattan),1873253,2.73


### `09_participacion_verdes.sql`

- **Pregunta:** Que peso tienen los taxis verdes dentro del total de viajes y esta cambiando?
- **Indicador:** % de los viajes de cada mes que hacen los taxis verdes.
- **Visualizacion:** linea.
- **Fuente:** trips_clean

In [12]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "09_participacion_verdes.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

0.50 s, 20 filas


,mes,pct_verdes
0,2024-01-01,1.811
1,2024-02-01,1.697
2,2024-03-01,1.539
3,2024-04-01,1.515
4,2024-05-01,1.552
5,2024-06-01,1.474
6,2024-07-01,1.594
7,2024-08-01,1.654
8,2024-09-01,1.438
9,2024-10-01,1.413


### `10_pago_sin_dato.sql`

- **Pregunta:** Que tan confiable es el dato del metodo de pago?
- **Indicador:** % de viajes sin metodo de pago informado (payment_type nulo o 0) por mes y tipo.
- **Visualizacion:** lineas.
- **Fuente:** trips_clean

In [13]:
df, seg = run_query_file(con, DIR_SQL / "07_indicadores" / "10_pago_sin_dato.sql")
print(f"{seg:.2f} s, {len(df)} filas")
df

0.64 s, 40 filas


,mes,taxi_type,pct_pago_sin_dato
0,2024-01-01,green,6.20
1,2024-01-01,yellow,4.02
2,2024-02-01,green,5.67
3,2024-02-01,yellow,5.14
4,2024-03-01,green,3.81
5,2024-03-01,yellow,10.64
6,2024-04-01,green,3.69
7,2024-04-01,yellow,11.40
8,2024-05-01,green,3.24
9,2024-05-01,yellow,10.69


![Tablero en Metabase](../docs/dashboard/tablero.png)